# Lab 4: Securely connect tools to your Agent with AgentCore Gateway 

## Overview

In this Lab, you will learn how to integrate tools available in your organization with the Customer Support Agent using the Amazon Bedrock Gateway.

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/docs/getting-started/intro) is an open protocol that standardizes how applications provide tools and context to Large Language Models (LLMs).

With [Amazon Bedrock Agent Core Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html), developers can convert APIs, Lambda functions, and existing services into MCP-compatible tools and make them available to agents through Gateway endpoints with just a few lines of code.

### Inbound and outbound authorization
Bedrock AgentCore Gateway provides secure connections via inbound and outbound authentication. For the inbound authentication, the AgentCore Gateway analyzes the OAuth token passed during invocation to decide allow or deny the access to a tool in the gateway. If a tool needs access to external resources, the AgentCore Gateway can use outbound authentication via API Key, IAM or OAuth Token to allow or deny the access to the external resource.

During the inbound authorization flow, an agent or the MCP client calls an MCP tool in the AgentCore Gateway adding an OAuth access token (generated from the user’s IdP). AgentCore Gateway then validates the OAuth access token and performs inbound authorization.

If the tool running in AgentCore Gateway needs to access external resources, OAuth will retrieve credentials of downstream resources using the resource credential provider for the Gateway target. AgentCore Gateway pass the authorization credentials to the caller to get access to the downstream API.

## Architecture

![Architecture Diagram](images/architecture_lab3_gateway.png)

### Key Features
- **Seamlessly integrate AWS Lambda functions:** This example shows how to integrate your Agent with existing AWS Lambda functions to check the warranty of an item and to get the customer profile using Amazon Bedrock AgentCore Gateway.
- **Secure your Gateway endpoint with Inbound Auth**: Only an Agent providing a valid JWT token can connect to the endpoint to use the tools
- **Configure the Agent to use the MCP endpoint**: The Agent gets a valid JWT token and uses it to connect to the MCP endpoint provided by AgentCore Gateway

## Prerequisites

* Python 3.10+
* AWS credentials configured
* Strands Agents and supporting libraries

## Step 1: Install and Import Required Libraries

In [1]:
# Install required packages
%pip install strands-agents "boto3>=1.39.15" python-dotenv strands-agents-tools bedrock_agentcore -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
import os
import sys
import boto3
import json
from bedrock_agentcore.identity.auth import requires_access_token
from mcp.client.streamable_http import streamablehttp_client
import requests

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../")))
from scripts.utils import get_ssm_parameter, put_ssm_parameter, load_api_spec, get_cognito_client_secret


sts_client = boto3.client('sts')

# Get AWS account details
REGION = boto3.session.Session().region_name

gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=REGION,
)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Implement Past Labs
NOTE: Please run the previous labs before starting this one. They will teach about their specific topics in depth and Lab 3's Google Auth steps are required for all following Labs

1. Lab 1. Four custom strands tools
2. Lab 2. AgentCore Memory
3. Lab 3. Identity

In [3]:
from lab_helpers.lab1_strands_agent import (
    get_return_policy,
    get_product_info,
)
from lab_helpers.lab2_memory import setup_memory, delete_memory

memory_hook = setup_memory()

## Step 3: Create a Lambda function to turn into an MCP server
In this step we see the code used by the AWS Lambda function to get customer profile of a given customer and to check the warranty status of a given item.

The Cloudformation used by this workshop already created the following resources used by this lab:
- AWS Lambda function which AgentCore Gateway exposes as an MCP-compatible endpoint
- AWS Lambda Execution IAM Role
- AgentCore Gateway IAM Role
- DynamoDB tables used by the AWS Lambda function. 
- Cognito User Pool and User Pool Client

AgentCore Gateway populates the Lambda context with the name of the tool to invoke, while the parameters passed to the tool are provided in the Lambda event:

```
extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
resource = extended_tool_name.split("___")[1]
```

In [ ]:
%%writefile lambda_function.py

from check_warranty import check_warranty_status
from get_customer_profile import get_customer_profile


def get_named_parameter(event, name):
    if name not in event:
        return None

    return event.get(name)


def lambda_handler(event, context):
    extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
    resource = extended_tool_name.split("___")[1]

    print(resource)

    if resource == "get_customer_profile":
        customer_id = get_named_parameter(event=event, name="customer_id")
        email = get_named_parameter(event=event, name="email")
        phone = get_named_parameter(event=event, name="phone")

        if not customer_id:
            return {
                "statusCode": 400,
                "body": "❌ Please provide customer_id",
            }

        try:
            customer_profile = get_customer_profile(
                customer_id=customer_id, email=email, phone=phone
            )
        except Exception as e:
            print(e)
            return {
                "statusCode": 400,
                "body": f"❌ {e}",
            }

        return {
            "statusCode": 200,
            "body": f"👤 Customer Profile Information: {customer_profile}",
        }

    elif resource == "check_warranty_status":
        serial_number = get_named_parameter(event=event, name="serial_number")
        customer_email = get_named_parameter(event=event, name="customer_email")

        if not serial_number:
            return {
                "statusCode": 400,
                "body": "❌ Please provide serial_number",
            }

        try:
            warranty_status = check_warranty_status(
                serial_number=serial_number, customer_email=customer_email
            )
        except Exception as e:
            print(e)
            return {
                "statusCode": 400,
                "body": f"❌ {e}",
            }

        return {
            "statusCode": 200,
            "body": warranty_status,
        }

    return {
        "statusCode": 400,
        "body": f"❌ Unknown toolname: {resource}",
    }


Overwriting lambda_function.py


## Step 4. Create your AgentCore Gateway

Now let's create the AgentCore Gateway to expose the Lambda function as MCP-compatible endpoint.

To validate the callers authorized to invoke our tools, we need to configure the Inbount Auth .

Inbound Auth works with OAuth authorization, where the client application must authenticate with the OAuth authorizer before using the Gateway. Your client would receive an access token which is used at runtime.

You need to specify an OAuth discovery server and client IDs. The Cloudformation provided with the workshop already provisioned the Cognito UserPool and UserPoolClient and it stored the discovery URL and the Client ID in dedicated SSM parameters.


In [5]:
gateway_name = "customersupport-gw"

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
        ],
        "discoveryUrl": get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
    }
}


print(f"Creating gateway in region {REGION} with name: {gateway_name}")

create_response = gateway_client.create_gateway(
    name=gateway_name,
    roleArn= get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="Customer Support AgentCore Gateway",
)

gateway_id = create_response["gatewayId"]

gateway = {
    "id": gateway_id,
    "name": gateway_name,
    "gateway_url": create_response["gatewayUrl"],
    "gateway_arn": create_response["gatewayArn"],
}
put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

print(f"✅ Gateway created successfully with ID: {gateway_id}")



Creating gateway in region us-east-1 with name: customersupport-gw
✅ Gateway created successfully with ID: customersupport-gw-goc8uyk0za


### Add the Lambda function Target

Then we need to write tool schema which describes the tools implemented by your Lambda function.

This file has been already defined in **TODO ADD LINK to the spec json file** 

In [6]:

def load_api_spec(file_path: str) -> list:
    with open(file_path, "r") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Expected a list in the JSON file")
    return data

try:
    api_spec_file = "./prerequisite/lambda/api_spec.json"

    # Validate API spec file exists
    if not os.path.exists(api_spec_file):
        print(f"❌ API specification file not found: {api_spec_file}")
        sys.exit(1)

    api_spec = load_api_spec(api_spec_file)

    # Use Cognito for Inbound OAuth to our Gateway
    lambda_target_config = {
        "mcp": {
            "lambda": {
                "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
                "toolSchema": {"inlinePayload": api_spec},
            }
        }
    }


    # Create gateway target
    credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaUsingSDK",
        description="Lambda Target using SDK",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=credential_config,
    )

    print(f"✅ Gateway target created: {create_target_response['targetId']}")

except Exception as e:
    print(f"❌ Error creating gateway target: {str(e)}")

✅ Gateway target created: BP8ULBHCAZ


## Step 5: Integrate Gateway with Strands Agent
Here we integrate our authentication token from Cognito into an MCPClient from Strands SDK to create an MCP Server object to integrate with our Strands Agent

In [7]:
def get_token(client_id: str, client_secret: str, scope_string: str, url: str) -> dict:
    try:
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        data = {
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": scope_string,

        }
        response = requests.post(url, headers=headers, data=data)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.RequestException as err:
        return {"error": str(err)}

In [8]:
gateway_access_token = get_token(
    get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    get_cognito_client_secret(),
    get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"))

print(f"Gateway Endpoint - MCP URL: {gateway['gateway_url']}")

# Set up MCP client
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway['gateway_url'],
        headers={"Authorization": f"Bearer {gateway_access_token['access_token']}"},
    )
)

Gateway Endpoint - MCP URL: https://customersupport-gw-goc8uyk0za.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp


## Step 6: Create and Configure the Customer Support Agent
Now we will create our Strands Agent using the AgentCore Gateway we built along with the resources from previous labs

In [9]:
# Initialize the Bedrock model
model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0"
model = BedrockModel(
    model_id=model_id,
    temperature=0.3,  # Balanced between creativity and consistency
    region_name=REGION
)

try:
    mcp_client.start()
except Exception as e:
    print(f"Error initializing agent: {str(e)}")

tools = (
            [
                get_product_info,
                get_return_policy,
            ]
            + mcp_client.list_tools_sync()
        )

# Create the customer support agent
agent = Agent(
    model=model,
    tools=tools,
    hooks=[memory_hook],
    system_prompt="""You are a helpful and professional customer support assistant for an e-commerce company.

Your role is to assist customers with:
- Order status inquiries
- Product information requests
- Shipping and delivery questions
- Return and refund policy questions

Guidelines for interactions:
- Always be polite, professional, and empathetic
- Use the available tools to provide accurate, up-to-date information
- If you cannot find specific information, acknowledge this and offer alternatives
- Keep responses clear and concise while being thorough
- If a customer has a complex issue that requires human intervention, politely suggest they contact our support team

Remember: Your goal is to provide excellent customer service and resolve customer inquiries efficiently."""
)

print("✅ Customer support agent created successfully!")

✅ Customer support agent created successfully!


## Step 7: Test the AgentGateway Supplemented Customer Support Agent

Let's test our agent with sample queries to ensure all features work correctly.

Note: You will see a message that says ```Polling for token for authorization url```, followed by a URL. Click on this URL to sign into your Google account, and give the agent the permissions to access your Google calendar.

In [10]:
test_prompts = [
    # Warranty Checks
    "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333.",
    "What are the warranty support guidelines ?"
]

# Function to test the agent
def test_agent_responses(agent, prompts):
    for i, prompt in enumerate(prompts, 1):
        print(f"\nTest Case {i}: {prompt}")
        print("-" * 50)
        try:
            response = agent(prompt)
        except Exception as e:
            print(f"Error: {str(e)}")
        print("-" * 50)

# Run the tests
test_agent_responses(agent, test_prompts)

print("\\n✅ Basic testing completed!")


Invalid search parameters: Memory resource is in a CREATING state.



Test Case 1: I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333.
--------------------------------------------------


Invalid search parameters: Memory resource is in a CREATING state.


I'll help you check the warranty status for your Gaming Console Pro device with serial number MNO33333333.
Tool #1: LambdaUsingSDK___check_warranty_status
I've checked the warranty status for your Gaming Console Pro device. Here's what I found:

**🛡️ Warranty Status Information**
- **Product:** Gaming Console Pro
- **Serial Number:** MNO33333333
- **Purchase Date:** November 25, 2023
- **Warranty End Date:** November 25, 2024
- **Warranty Type:** Gaming Warranty
- **Status:** ❌ **Expired** (expired 259 days ago)

**Coverage Details:**
Your warranty covered controller issues, overheating protection, and hard drive replacement.

**What's Next:**
Since your warranty has expired, you may still have options:
- Extended warranty options may be available
- We can provide repair service pricing for any issues you're experiencing

If you need assistance with repairs or want to explore extended warranty options, I'd recommend contacting our support team directly. They can provide you with curren

Failed to create event: An error occurred (ValidationException) when calling the CreateEvent operation: Memory status is not active, unable to process CreateEvent request
Failed to save support interaction: An error occurred (ValidationException) when calling the CreateEvent operation: Memory status is not active, unable to process CreateEvent request


 about our repair services?--------------------------------------------------

Test Case 2: What are the warranty support guidelines ?
--------------------------------------------------


Invalid search parameters: Memory resource is in a CREATING state.
Invalid search parameters: Memory resource is in a CREATING state.


I'll get the detailed warranty support guidelines for your Gaming Console Pro device.
Tool #2: get_product_info
Let me try getting general gaming console warranty information:
Tool #3: get_product_info
I apologize, but I don't have access to the specific warranty support guidelines for Gaming Console Pro devices in my current system. However, I can provide you with some general information about what warranty support guidelines typically include:

**Common Warranty Support Guidelines Usually Cover:**

🔧 **What's Covered:**
- Manufacturing defects
- Hardware malfunctions
- Component failures under normal use

📋 **Support Process:**
- How to initiate warranty claims
- Required documentation (proof of purchase, serial number)
- Diagnostic procedures
- Repair or replacement procedures

⏰ **Timeframes:**
- Response times for warranty claims
- Repair completion estimates
- Replacement processing times

📞 **Contact Methods:**
- Warranty support phone numbers
- Online claim submission processe

Failed to create event: An error occurred (ValidationException) when calling the CreateEvent operation: Memory status is not active, unable to process CreateEvent request
Failed to save support interaction: An error occurred (ValidationException) when calling the CreateEvent operation: Memory status is not active, unable to process CreateEvent request


--------------------------------------------------
\n✅ Basic testing completed!


## Step 8: Cleanup (Optional)
This section cleans up all of the resources from this lab. This will prevent your AWS account from having old, unused resources.

In [12]:
from lab_helpers.lab2_memory import delete_memory

print("🗑️ Deleting Memory Hook")
delete_memory(memory_hook)
print(f"✅ Memory hook deleted")

print(f"🗑️  Deleting all targets for gateway: {gateway_id}")

# List and delete all targets
list_response = gateway_client.list_gateway_targets(
    gatewayIdentifier=gateway_id, maxResults=100
)

for item in list_response["items"]:
    target_id = item["targetId"]
    print(f"   Deleting target: {target_id}")
    gateway_client.delete_gateway_target(
        gatewayIdentifier=gateway_id, targetId=target_id
    )
    print(f"   ✅ Target {target_id} deleted")

# Delete the gateway
print(f"🗑️  Deleting gateway: {gateway_id}")
gateway_client.delete_gateway(gatewayIdentifier=gateway_id)
print(f"✅ Gateway {gateway_id} deleted successfully")

🗑️ Deleting Memory Hook
✅ Memory hook deleted
🗑️  Deleting all targets for gateway: customersupport-gw-goc8uyk0za
   Deleting target: BP8ULBHCAZ
   ✅ Target BP8ULBHCAZ deleted
🗑️  Deleting gateway: customersupport-gw-goc8uyk0za
✅ Gateway customersupport-gw-goc8uyk0za deleted successfully


## Congratulations! 🎉

You have successfully completed **Lab 3: Securely connect tools to your Agent with AgentCore Gateway**!

### What You Accomplished:

✅ **Added 2 New Customer Support Tools**: using AgentCore Gateway to expose an AWS Lambda Function via an MCP endpoint

✅ **Integrated AgentCore Gateway with the customer support Agent**: You used Lambda and Strands to create an MCP Server that could be hosted via AgentCore Gateway

✅ **Outbound Authentication on your Gateway**: You used AgentCore Identity to connect Cognito authentication to your AgentCore Gateway

## Resources
- [Amazon Bedrock Agent Core Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)
- [Strands Agents Documentation](https://github.com/strands-agents/sdk-python)
- [Official Customer Support Sample](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/02-use-cases/customer-support-assistant)
